# 02 — GPU workloads: FP32 versus FP16

**Question:** what changes when the same model runs in FP16 instead of FP32 on the same GPU?

Use a **fresh Colab T4 GPU session** and run cells in order. This lesson requires CUDA and never falls back to CPU inference. Keep the T4, model revision, prompts, and generation settings fixed throughout the comparison.

We use the same `LiquidAI/LFM2.5-1.2B-Instruct` revision as lesson 01. FP32 stores a floating-point weight in four bytes; FP16 uses two. Some operations may still use higher precision internally. Predict how parameter storage, peak memory, generation latency, and output text will change. Faster generation is a hypothesis to measure.

The suite contains three synthetic input lengths. These are controlled length probes, not a representative quality evaluation. Baseline timing and token diagnostics run separately. Default settings produce 96 requests including warmups; they are exploratory measurements, not a reliable p95/p99 study.

## 1. Install and restart if necessary

Use the GPU runtime's compatible CUDA-enabled PyTorch. If this session already imported an older Transformers version, restart the session after installation and continue from section 2. A fresh session also prevents a model from lesson 01 from inflating this lesson's memory measurements.

In [1]:
# Install dependencies in this kernel; retain the GPU runtime's compatible PyTorch build.
%pip install -q "torch>=2.6,<3" "transformers==5.3.0" psutil

## 2. Fix the experiment configuration and verify the GPU

All timing values saved below use **seconds**; memory uses **GiB**. The baseline measures generation without token callbacks. The diagnostic pass additionally measures TTFT, TPOT, and inter-token gaps.

Leave the defaults for the first comparison. FP32 runs before FP16; warmups reduce first-use effects but do not remove all ordering effects. Reverse `PRECISIONS` in a later independent run to check an apparent improvement. Changing repetition counts creates a new recorded configuration.

In [2]:
import csv  # Export request and comparison tables.
import gc  # Release Python references before freeing cached GPU memory.
import json  # Preserve complete configurations, responses, and token timelines.
import os  # Record host CPU resources used to feed GPU execution.
import platform  # Identify the interpreter and operating system.
import shutil  # Package all result files into one downloadable archive.
import statistics  # Summarize repeated measurements with medians.
import subprocess  # Read driver and GPU information through nvidia-smi.
import sys  # Detect stale imported packages after an upgrade.
import time  # Measure wall-clock generation and token arrivals.
import uuid  # Make output directories unique.
from datetime import datetime, timezone  # Label runs with an explicit UTC timestamp.
from importlib.metadata import version  # Read installed dependency versions.
from pathlib import Path  # Build output and metadata paths.

# Catch the stale-import problem encountered in lesson 01 before loading a model.
for module_name, package_name in [("transformers", "transformers"), ("huggingface_hub", "huggingface-hub")]:
    loaded = sys.modules.get(module_name)  # Inspect any already imported module.
    if loaded is not None and loaded.__version__ != version(package_name):
        raise RuntimeError("Restart the session/kernel after the package upgrade, then continue from section 2.")

import psutil  # Record host RAM capacity.
import torch  # Execute the model on CUDA.
import transformers  # Record the loaded Transformers version.
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig  # Load weights/tokenizer and configure greedy generation.
from transformers.generation.streamers import BaseStreamer  # Observe generated IDs without word buffering.

MODEL_NAME = "LiquidAI/LFM2.5-1.2B-Instruct"  # Keep the model from the first lesson.
MODEL_REVISION = "b9d816ab510f9a28dca37cb182e9da8b0226e13c"  # Pin the exact previously measured checkpoint.
PRECISIONS = ["float32", "float16"]  # Run matched workloads in this order.
MAX_NEW_TOKENS = 64  # Keep the same output budget across precision settings.
WARMUP_RUNS = 2  # Warm each baseline workload before measuring it.
MEASURED_RUNS = 10  # Increase the sample count beyond lesson 01.
DIAGNOSTIC_RUNS = 3  # Use a separate small pass for token-level measurements; set 0 to disable it.
SEED = 42  # Record a stable seed; numerical behavior can still differ across precision.

if not torch.cuda.is_available():  # Avoid silently running a GPU experiment on CPU.
    raise RuntimeError("CUDA GPU required. Select a GPU runtime in Colab and reconnect with a CUDA-enabled PyTorch build.")
assert PRECISIONS and len(set(PRECISIONS)) == len(PRECISIONS) and set(PRECISIONS) <= {"float32", "float16"}
assert MAX_NEW_TOKENS > 0 and WARMUP_RUNS >= 1 and MEASURED_RUNS >= 1 and DIAGNOSTIC_RUNS >= 0

device = torch.device("cuda:0")  # Use one GPU consistently throughout the experiment.
gc.collect()  # Collect abandoned objects from earlier interactive cells.
torch.cuda.empty_cache()  # Release unused allocator reservations before checking the starting state.
if torch.cuda.memory_allocated(device) > 0:  # Detect tensors or a model left on this process's GPU.
    raise RuntimeError("GPU tensors already exist in this kernel. Restart the GPU session before this lesson for clean memory measurements.")
torch.manual_seed(SEED)  # Fix the seed before loading and generation.
torch.set_float32_matmul_precision("highest")  # Keep FP32 matrix multiplication at full precision.
torch.backends.cudnn.allow_tf32 = False  # Also disable reduced-precision TF32 convolution in the FP32 comparison.

try:
    gpu_inventory = subprocess.check_output(["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total,memory.free,power.limit", "--format=csv"], text=True).strip()  # Record driver, device capacity, and a free-memory snapshot.
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    gpu_inventory = f"Unavailable: {type(error).__name__}"  # Keep missing inventory explicit without hiding a usable CUDA GPU.

properties = torch.cuda.get_device_properties(device)  # Read the GPU selected by PyTorch.
free_bytes, total_bytes = torch.cuda.mem_get_info(device)  # Device-wide snapshot, not peak process memory.
environment = {
    "python": platform.python_version(),  # Record the interpreter used by this run.
    "packages": {name: version(name) for name in ["torch", "transformers", "huggingface-hub", "tokenizers", "safetensors", "psutil"]},  # Pin the observed software environment in the artifact.
    "loaded_transformers_version": transformers.__version__,  # Compare loaded code with installed metadata.
    "platform": platform.platform(),  # Record OS, architecture, and libc identification.
    "logical_cpu_count": os.cpu_count(),  # Record host resources without running CPU benchmarks.
    "torch_cpu_threads": torch.get_num_threads(),  # Record host tensor-operation threading.
    "system_ram_gib": psutil.virtual_memory().total / 2**30,  # Record host RAM capacity, not process usage.
    "device": str(device), "gpu_name": properties.name,  # Identify the actual accelerator.
    "compute_capability": list(torch.cuda.get_device_capability(device)),  # Record the GPU's hardware generation capability.
    "gpu_total_gib": total_bytes / 2**30, "gpu_free_gib_at_start": free_bytes / 2**30,  # Record device capacity and the initial snapshot separately.
    "cuda_build": torch.version.cuda, "cudnn_version": torch.backends.cudnn.version(),  # Record library build versions rather than inferring the driver.
    "float32_matmul_precision": torch.get_float32_matmul_precision(), "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,  # Record numerical execution policies.
    "nvidia_smi_inventory": gpu_inventory,  # Preserve the driver and power-limit report when available.
}
print(json.dumps(environment, indent=2))  # Inspect hardware before spending time on inference.

{
  "python": "3.13.16",
  "packages": {
    "torch": "2.14.1+cu130",
    "transformers": "5.3.0",
    "huggingface-hub": "1.33.0",
    "tokenizers": "0.22.2",
    "safetensors": "0.8.0",
    "psutil": "5.9.5"
  },
  "loaded_transformers_version": "5.3.0",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "logical_cpu_count": 8,
  "torch_cpu_threads": 4,
  "system_ram_gib": 50.99027633666992,
  "device": "cuda:0",
  "gpu_name": "Tesla T4",
  "compute_capability": [
    7,
    5
  ],
  "gpu_total_gib": 14.56317138671875,
  "gpu_free_gib_at_start": 14.46075439453125,
  "cuda_build": "13.0",
  "cudnn_version": 92800,
  "float32_matmul_precision": "highest",
  "cudnn_allow_tf32": false,
  "nvidia_smi_inventory": "index, name, driver_version, memory.total [MiB], memory.free [MiB], power.limit [W]\n0, Tesla T4, 580.82.07, 15360 MiB, 14910 MiB, 70.00 W"
}


## 3. Prepare a fixed workload suite

Every prompt asks the same question over repetitions of the same fictional service note. Repetition changes input length without introducing new facts. The three sizes are measured after chat formatting, rather than assumed from character counts. This synthetic suite explores length and precision; it is not enough to judge general model quality.

Prompts remain on CPU until their case runs. Tokenization and input transfer are excluded from the generation timer. Their exact text and IDs are saved once in the JSON.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)  # Use one pinned tokenizer for both precisions.
note = "The service runs one language model on a GPU. Requests wait in a queue. Batching improves throughput but can increase waiting time. Longer inputs use more memory. The operator measures latency and errors."  # Use fictional, repeatable benchmark content.
question = "Summarize the service notes in exactly three short sentences."  # Keep the task fixed across lengths.
workloads = []  # Preserve workload IDs, text, and tokenized CPU tensors.
for prompt_id, copies in [("short", 1), ("medium", 8), ("long", 32)]:
    prompt = "Service notes:\n" + "\n".join([note] * copies) + "\n\n" + question  # Change the amount of context while retaining the same facts.
    rendered = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)  # Apply the model's chat format.
    inputs = tokenizer(rendered, add_special_tokens=False, return_tensors="pt")  # Avoid adding template special tokens twice.
    workloads.append({"prompt_id": prompt_id, "prompt": prompt, "rendered_prompt": rendered, "input_token_ids": inputs["input_ids"][0].tolist(), "inputs": inputs})  # Keep inputs out of timed generation work.
    print(prompt_id, "input tokens:", inputs["input_ids"].shape[1])  # Check actual workload sizes.

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.49k [00:00<?, ?B/s]

short input tokens: 66
medium input tokens: 353
long input tokens: 1337


## 4. Define what a request measures

The baseline timer surrounds `generate()` and GPU synchronization. It includes prefill, decoding, and generation-loop overhead; it excludes loading, tokenization, input transfer, final text decoding, and printing.

The diagnostic pass uses the same token-ID streamer method as lesson 01. It skips the initial prompt callback and timestamps generated IDs after Transformers copies them to CPU. **TTFT** measures the first arrival. **TPOT** is `(generation_seconds - ttft_seconds) / (output_tokens - 1)`, including completion cleanup. **ITL** values are adjacent arrival gaps; their mean excludes cleanup. TPOT/mean ITL are undefined for fewer than two generated tokens. EOS is included in output counts.

Stream callbacks add overhead, so baseline and diagnostic results are never pooled. These are in-process timings with no HTTP server or request queue.

Allocated memory measures live tensors; reserved memory includes the allocator's cached blocks. Peak counters are reset before each request, retaining the loaded model allocation. The device-wide used-memory value is a snapshot after completion, not a peak or utilization measurement.

In [4]:
class TokenTimingStreamer(BaseStreamer):
    # Record token IDs as they reach Python, without waiting for decoded words.
    def __init__(self):
        self.skip_prompt = True  # The first callback contains the complete input prompt.
        self.arrivals = []  # Preserve one wall-clock timestamp per generated token.

    def put(self, value):
        arrived = time.perf_counter()  # Timestamp callback entry after the CPU copy.
        if self.skip_prompt:
            self.skip_prompt = False  # Exclude the prompt even when it contains only one token.
            return
        if value.numel() != 1:  # This lesson supports one greedy request at a time.
            raise ValueError("Expected one generated token per callback.")
        self.arrivals.append(arrived)  # Preserve timing without decoding text inside the timer.

    def end(self):
        pass  # No background queue or thread needs closing.


def measure_request(model, inputs, generation_config, diagnostic=False):
    # Return measurements and a complete response, without retaining output GPU tensors.
    streamer = TokenTimingStreamer() if diagnostic else None  # Leave the baseline uninstrumented.
    torch.cuda.synchronize(device)  # Exclude previously queued GPU work.
    torch.cuda.reset_peak_memory_stats(device)  # Reset peaks without clearing the loaded model or allocator.
    start = time.perf_counter()  # Start immediately before generation.
    with torch.inference_mode():  # Disable gradient bookkeeping.
        output = model.generate(**inputs, generation_config=generation_config, streamer=streamer)  # Generate one continuation with a fresh request cache.
    torch.cuda.synchronize(device)  # Wait for completed GPU work before stopping the timer.
    elapsed = time.perf_counter() - start  # Record completed generation latency.
    input_tokens = inputs["input_ids"].shape[1]  # Read the prompt length from the actual tensors.
    generated_ids = output[0, input_tokens:].cpu().tolist()  # Remove the prompt and preserve exact output IDs.
    eos = generation_config.eos_token_id  # Inspect the configured termination IDs.
    eos_ids = [eos] if isinstance(eos, int) else eos or []  # Normalize a single ID or list.
    free_bytes, total_bytes = torch.cuda.mem_get_info(device)  # Sample device-wide usage after generation.
    row = {
        "input_tokens": input_tokens, "output_tokens": len(generated_ids),  # Record actual work, including any terminal EOS token.
        "generation_seconds": elapsed, "output_tokens_per_second": len(generated_ids) / elapsed,  # Keep unrounded request measurements.
        "peak_allocated_gib": torch.cuda.max_memory_allocated(device) / 2**30,  # Includes loaded model and request tensor allocations.
        "peak_reserved_gib": torch.cuda.max_memory_reserved(device) / 2**30,  # Includes allocator reservations carried into the request.
        "device_used_gib_after": (total_bytes - free_bytes) / 2**30,  # Includes other processes/context allocations; this is a snapshot.
        "finish_reason": "eos" if generated_ids and generated_ids[-1] in eos_ids else "length" if len(generated_ids) == MAX_NEW_TOKENS else "other",  # Distinguish natural completion from a capped answer.
        "generated_token_ids": generated_ids, "response": tokenizer.decode(generated_ids, skip_special_tokens=True),  # Preserve the response outside the generation timer.
    }
    if diagnostic:
        arrivals = [arrived - start for arrived in streamer.arrivals]  # Convert timestamps to elapsed seconds.
        if len(arrivals) != len(generated_ids):  # Detect missed or grouped token callbacks.
            raise RuntimeError("Token timestamp count differs from generated output length.")
        gaps = [later - earlier for earlier, later in zip(arrivals, arrivals[1:])]  # Calculate all N-1 adjacent gaps.
        row.update({
            "ttft_seconds": arrivals[0] if arrivals else None,  # Time until the first token ID reaches Python.
            "tpot_seconds": (elapsed - arrivals[0]) / (len(arrivals) - 1) if len(arrivals) > 1 else None,  # Average post-first-token time through completion.
            "mean_itl_seconds": statistics.mean(gaps) if gaps else None,  # Average observed token gaps without final cleanup.
            "min_itl_seconds": min(gaps) if gaps else None, "max_itl_seconds": max(gaps) if gaps else None,  # Preserve the range of within-request gaps.
            "token_arrival_seconds": arrivals, "inter_token_latency_seconds": gaps,  # Retain full timelines in JSON.
        })
    return row  # All returned values are ordinary Python data.

## 5. Create a run record before benchmarking

Results are checkpointed to JSON **after every request**, outside the generation timer. If a request fails, its error is retained and the run stops. Partial runs remain labeled incomplete. No synthetic benchmark results are provided.

The output directory is unique and ignored by Git. Preserve selected reviewed files in the repository's `results/` directory after interpretation. The full JSON includes prompts, token IDs, settings, responses, memory metrics, warmups, and measured requests.

In [5]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]  # Use UTC and a unique suffix for this run.
output_dir = Path("runs") / "gpu-benchmark" / run_id  # Keep each run's artifacts together.
output_dir.mkdir(parents=True, exist_ok=False)  # Avoid overwriting a prior experiment.
json_path = output_dir / "results.json"  # Persist the complete record incrementally.
records = []  # Retain successful and failed requests, including warmups.
payload = {
    "benchmark_version": "gpu-precision-v1", "run_id": run_id, "complete": False,  # Keep incomplete runs distinguishable.
    "environment": environment, "model": MODEL_NAME, "model_revision": MODEL_REVISION,  # Record observed hardware and fixed checkpoint identity.
    "settings": {"precision_order": PRECISIONS, "max_new_tokens": MAX_NEW_TOKENS, "warmups_per_baseline": WARMUP_RUNS, "measured_runs": MEASURED_RUNS, "diagnostic_runs": DIAGNOSTIC_RUNS, "seed": SEED, "batch_size": 1},  # Preserve the requested procedure.
    "timing_scope": "model.generate plus CUDA synchronization; excludes tokenization, input transfer and final text decoding",  # Define the generation boundary.
    "diagnostic_definitions": {"ttft": "first token ID received by Python after generate start", "tpot": "(generation_seconds - ttft_seconds) / (output_tokens - 1)", "itl": "gap between consecutive token ID arrivals"},  # Keep metric definitions with the data.
    "workloads": [{key: value for key, value in case.items() if key != "inputs"} for case in workloads],  # Save exact prompts/IDs without serializing tensors.
    "models": {}, "requests": records,  # Add per-precision loading/configuration and request records during execution.
}

def save_run():
    # Keep serialization outside the generation timer and reject non-finite metric values.
    json_path.write_text(json.dumps(payload, indent=2, allow_nan=False) + "\n", encoding="utf-8")

save_run()  # Persist configuration even if model loading later fails.
print("Run directory:", output_dir.resolve())  # Locate the artifacts in the current runtime.

Run directory: /content/runs/gpu-benchmark/20261122T214341Z-f97511de


## 6. Run FP32, release its memory, then run FP16

For each precision, the notebook loads one model, runs each prompt's baseline warmups and measurements, then its separate diagnostic warmup and measurements. Earlier request caches are not reused as answer caches. No concurrent requests are sent.

The previous model is deleted and unused allocator blocks are released between precisions. We do not clear the allocator between requests. Reserved-memory peaks can therefore depend on earlier workload sizes; both precisions use the same workload order. Loading time is recorded separately.

**Before running:** predict the memory change. After running, inspect actual output lengths before interpreting latency ratios. A lower latency with fewer output tokens is partly a workload change. Unsupported operations or runtime failures are saved and surfaced rather than silently excluded.

In [6]:
def benchmark_precision(precision):
    # Limit model/tensor lifetimes to this precision so the next model can start cleanly.
    model = None  # Make cleanup safe even when loading fails.
    try:
        started = time.perf_counter()  # Keep loading outside request measurements.
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, dtype=getattr(torch, precision), attn_implementation="sdpa").to(device)  # Load one pinned model at the selected precision.
        model.eval()  # Disable training-specific behavior.
        torch.cuda.synchronize(device)  # Finish device placement before recording setup time.
        config = GenerationConfig(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1, use_cache=True, eos_token_id=model.generation_config.eos_token_id, pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id)  # Keep generation settings matched across precision.
        payload["models"][precision] = {"load_seconds": time.perf_counter() - started, "actual_dtype": str(next(model.parameters()).dtype), "parameter_count": sum(p.numel() for p in model.parameters()), "parameter_gib": sum(p.numel() * p.element_size() for p in model.parameters()) / 2**30, "attention_implementation": model.config._attn_implementation, "generation_config": config.to_dict()}  # Record the actual model configuration.
        print("Loaded", precision, json.dumps({key: value for key, value in payload["models"][precision].items() if key != "generation_config"}))  # Inspect weights before requests run.
        save_run()  # Preserve successful loading before starting workloads.
        for case in workloads:
            inputs = {key: value.to(device) for key, value in case["inputs"].items()}  # Copy inputs without mutating the original CPU workload tensors.
            torch.cuda.synchronize(device)  # Exclude input transfer from generation timing.
            modes = [("baseline", WARMUP_RUNS, MEASURED_RUNS)] + ([("diagnostic", 1, DIAGNOSTIC_RUNS)] if DIAGNOSTIC_RUNS else [])  # Keep instrumented and baseline samples separate.
            for mode, warmups, repeats in modes:
                for phase, count in [("warmup", warmups), ("measured", repeats)]:
                    for repeat in range(1, count + 1):
                        identity = {"precision": precision, "prompt_id": case["prompt_id"], "mode": mode, "phase": phase, "repeat": repeat}  # Identify every request uniquely within the run.
                        try:
                            row = {**identity, "status": "ok", **measure_request(model, inputs, config, diagnostic=mode == "diagnostic")}  # Collect metrics and full response data.
                        except Exception as error:
                            records.append({**identity, "status": "error", "error_type": type(error).__name__, "error": str(error)})  # Retain failures without fabricating timing values.
                            save_run()  # Preserve the partial run before surfacing the exception.
                            raise
                        records.append(row)  # Keep warmups as well as successful measured requests.
                        save_run()  # Write outside the generation timer after every completed request.
                print(precision, case["prompt_id"], mode, "complete")  # Show progress without printing inside timed generation.
            del inputs  # Release this workload's GPU input tensors.
    except Exception as error:
        payload["fatal_error"] = {"precision": precision, "type": type(error).__name__, "message": str(error)}  # Record loading or request failures at run level too.
        save_run()  # Leave the run labeled incomplete.
        raise
    finally:
        del model  # Drop this precision's model reference before loading another model.
        gc.collect()  # Collect Python objects that might retain GPU tensors.
        torch.cuda.empty_cache()  # Release unused allocator reservations between precisions.

for precision in PRECISIONS:
    benchmark_precision(precision)  # Run each precision sequentially on the same GPU.
payload["complete"] = True  # Mark completion only after every configured workload succeeds.
save_run()  # Persist completion independently of the later display/export cells.

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded float32 {"load_seconds": 15.363551631000064, "actual_dtype": "torch.float32", "parameter_count": 1170340608, "parameter_gib": 4.359858512878418, "attention_implementation": "sdpa"}
float32 short baseline complete
float32 short diagnostic complete
float32 medium baseline complete
float32 medium diagnostic complete
float32 long baseline complete
float32 long diagnostic complete


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded float16 {"load_seconds": 1.363913211999943, "actual_dtype": "torch.float16", "parameter_count": 1170340608, "parameter_gib": 2.179929256439209, "attention_implementation": "sdpa"}
float16 short baseline complete
float16 short diagnostic complete
float16 medium baseline complete
float16 medium diagnostic complete
float16 long baseline complete
float16 long diagnostic complete


## 7. Summarize matched cases and inspect responses

Only successful **measured** requests contribute to medians. Warmups stay in the raw record. The table includes sample counts and output-length ranges. No p95/p99 claim is made from these small samples.

Compare FP32 and FP16 within the **same prompt ID and timing mode**. Token-ID/text equality is a useful change detector, not a quality score. Inspect factual content, the requested sentence count, truncation, and any unexpected output. A changed answer may still be correct; an unchanged answer can still be wrong.

In [7]:
summaries = []  # Keep each precision/workload/mode as a separate measurement group.
metrics = ["generation_seconds", "output_tokens_per_second", "peak_allocated_gib", "peak_reserved_gib", "ttft_seconds", "tpot_seconds", "mean_itl_seconds"]  # Summarize only metrics actually recorded in each mode.
for precision in PRECISIONS:
    for case in workloads:
        for mode in ["baseline"] + (["diagnostic"] if DIAGNOSTIC_RUNS else []):
            attempts = [row for row in records if row["precision"] == precision and row["prompt_id"] == case["prompt_id"] and row["mode"] == mode and row["phase"] == "measured"]  # Exclude warmups from this summary.
            good = [row for row in attempts if row["status"] == "ok"]  # Report failures separately rather than silently treating them as successes.
            summary = {"precision": precision, "prompt_id": case["prompt_id"], "mode": mode, "attempted_requests": len(attempts), "successful_requests": len(good), "failed_requests": len(attempts) - len(good), "input_tokens": len(case["input_token_ids"]), "min_output_tokens": min((row["output_tokens"] for row in good), default=None), "max_output_tokens": max((row["output_tokens"] for row in good), default=None)}  # Retain the workload lengths and counts.
            for metric in metrics:
                values = [row[metric] for row in good if row.get(metric) is not None]  # Leave unavailable or undefined metrics null.
                summary[f"median_{metric}"] = statistics.median(values) if values else None  # Preserve unrounded medians for export.
            summaries.append(summary)  # Store the group before printing it.
            print(json.dumps(summary))  # Inspect latency, output rate, memory, and diagnostic medians.

for case in workloads:
    print("\nRESPONSE COMPARISON:", case["prompt_id"])  # Inspect the same task across precisions.
    examples = [next((row for row in records if row["precision"] == precision and row["prompt_id"] == case["prompt_id"] and row["mode"] == "baseline" and row["phase"] == "measured" and row["status"] == "ok"), None) for precision in PRECISIONS]  # Choose a real measured response per precision.
    for precision, example in zip(PRECISIONS, examples):
        if example is not None:
            print(precision, "tokens:", example["output_tokens"], "finish:", example["finish_reason"], "\n", example["response"])  # Compare full text and whether the token cap was reached.
    if len(examples) == 2 and all(example is not None for example in examples):
        print("Identical generated IDs:", examples[0]["generated_token_ids"] == examples[1]["generated_token_ids"])  # Detect precision-dependent output changes without assigning a quality score.
payload["summaries"] = summaries  # Keep summaries alongside the raw records used to compute them.
save_run()  # Persist the summary without replacing individual requests.

{"precision": "float32", "prompt_id": "short", "mode": "baseline", "attempted_requests": 10, "successful_requests": 10, "failed_requests": 0, "input_tokens": 66, "min_output_tokens": 41, "max_output_tokens": 41, "median_generation_seconds": 1.0536100819999774, "median_output_tokens_per_second": 38.91383296199035, "median_peak_allocated_gib": 4.377768516540527, "median_peak_reserved_gib": 4.38671875, "median_ttft_seconds": null, "median_tpot_seconds": null, "median_mean_itl_seconds": null}
{"precision": "float32", "prompt_id": "short", "mode": "diagnostic", "attempted_requests": 3, "successful_requests": 3, "failed_requests": 0, "input_tokens": 66, "min_output_tokens": 41, "max_output_tokens": 41, "median_generation_seconds": 1.0671811130000606, "median_output_tokens_per_second": 38.41897078251391, "median_peak_allocated_gib": 4.377768516540527, "median_peak_reserved_gib": 4.38671875, "median_ttft_seconds": 0.07270401499999934, "median_tpot_seconds": 0.024853411075002895, "median_mean_i

## 8. Export, print, and download

`results.json` contains the complete run, including responses and token timelines. `requests.csv` keeps every warmup/measured request, status/error, full response, and scalar metrics. `summary.csv` contains grouped medians and sample counts.

The CSV files are printed below. Set `PRINT_FULL_JSON = True` to print the complete JSON in the notebook too. Save the executed notebook. The optional download cell creates one ZIP to avoid multiple browser downloads; run it in the Colab browser before replacing the runtime.

In [8]:
def write_table(path, rows):
    # Use the union of fields so diagnostic metrics and error fields are not discarded.
    columns = list(dict.fromkeys(key for row in rows for key in row if key not in ["generated_token_ids", "token_arrival_seconds", "inter_token_latency_seconds"]))  # Keep large arrays in JSON only.
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=columns, extrasaction="ignore")  # Quote full responses, including embedded newlines and commas.
        writer.writeheader()  # Name the request/summary columns.
        writer.writerows(rows)  # Preserve every provided record, including failures.

requests_path = output_dir / "requests.csv"  # Export individual request data separately from aggregates.
summary_path = output_dir / "summary.csv"  # Export comparable precision/workload/mode summaries.
write_table(requests_path, records)  # Keep baseline and diagnostic rows explicitly labeled.
write_table(summary_path, summaries)  # Preserve the aggregate table used in the analysis.
PRINT_FULL_JSON = True  # Keep complete results visible for later inspection from the saved notebook.
for path in [json_path, requests_path, summary_path]:
    print(f"\n{path.name}: {path.stat().st_size / 1024:.1f} KiB | {path.resolve()}")  # Report actual artifact sizes and runtime locations.
    if path.suffix != ".json" or PRINT_FULL_JSON:
        print(path.read_text(encoding="utf-8"))  # Print saved artifacts rather than recomputing any measurements.


results.json: 235.5 KiB | /content/runs/gpu-benchmark/20261122T214341Z-f97511de/results.json
{
  "benchmark_version": "gpu-precision-v1",
  "run_id": "20261122T214341Z-f97511de",
  "complete": true,
  "environment": {
    "python": "3.13.16",
    "packages": {
      "torch": "2.14.1+cu130",
      "transformers": "5.3.0",
      "huggingface-hub": "1.33.0",
      "tokenizers": "0.22.2",
      "safetensors": "0.8.0",
      "psutil": "5.9.5"
    },
    "loaded_transformers_version": "5.3.0",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "logical_cpu_count": 8,
    "torch_cpu_threads": 4,
    "system_ram_gib": 50.99027633666992,
    "device": "cuda:0",
    "gpu_name": "Tesla T4",
    "compute_capability": [
      7,
      5
    ],
    "gpu_total_gib": 14.56317138671875,
    "gpu_free_gib_at_start": 14.46075439453125,
    "cuda_build": "13.0",
    "cudnn_version": 92800,
    "float32_matmul_precision": "highest",
    "cudnn_allow_tf32": false,
    "nvidia_smi_inventory": "inde

In [ ]:
# Optional, Colab only: uncomment to download all files in this run as one ZIP.
# from google.colab import files  # Access the browser download helper.
# archive = shutil.make_archive(str(output_dir), "zip", output_dir)  # Create the ZIP beside the run directory.
# files.download(archive)  # Download it before replacing the runtime.

## 9. Interpret before changing another variable

Record your findings in a separate GPU precision report after the run:

- Did parameter storage and peak allocated/reserved memory change by the same proportion? Why might they differ?
- For each prompt, did FP16 change latency and output rate? Did output lengths or stop reasons change too?
- Did TTFT and TPOT improve equally? Remember that TTFT includes more than pure prefill.
- What did the token callbacks change relative to the baseline? Compare modes separately.
- Were answers factually useful, complete, and in the requested format? Exact equality alone is not a quality evaluation.
- Did the longer inputs change memory, TTFT, or completion behavior?
- Could fixed precision order, first-use effects, cached allocator blocks, or Colab load affect the result?
- Which conclusion should be checked with another run or a larger prompt set?

The next experiments isolate batching and GPU memory behavior, then introduce a serving engine. Keep this lesson's batch size at one.

Sources: [model card](https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct), [native model implementation](https://github.com/huggingface/transformers/blob/v5.3.0/src/transformers/models/lfm2/modeling_lfm2.py), [token delivery](https://github.com/huggingface/transformers/blob/v5.3.0/src/transformers/generation/utils.py), [CUDA timing and memory](https://docs.pytorch.org/docs/stable/notes/cuda.html).